# DEPICT random-split prediction extraction

This notebook extracts, for every requested random-split partition, the exact matched baseline profile used by DEPICT (`x_base`), the observed post-treatment profile (`y_true`), and the model-predicted post-treatment profile (`logits`). Each row is aligned through `row_in_prediction_file`.

In [ ]:
# -----------------------------------------------------------------------------
# 0. Configuration: edit these paths only if your Longleaf layout changes.
# -----------------------------------------------------------------------------
from pathlib import Path

DATA_PATH = Path("~/DEPICT/Data/FinalData/adataAfterClean.h5ad")
LLM_PATH = Path("~/DEPICT/Data/FinalData/gptEmbed_Jul9_final.csv")
MFP_PATH = Path("~/DEPICT/Data/FinalData/compounds_512MFP_wholeDat_fixed.csv")
TARGET_PATH = Path("~/DEPICT/Data/FinalData/compounds_target_multihot_full.csv")
MODEL_DIR = Path("~/DEPICT/Model")

OUT_DIR = Path("~/DEPICT/Code/downstream_analysis_code/DGElandscape/analysis/resp_latents_random_split")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SPLITS = [f"random_split{i}" for i in range(1, 6)]

# Default recommendation for the XPert-like UMAP: held-out profiles only.
# Change to ("train", "valid", "test") only when you intentionally want
# embeddings for all profiles assigned under each Monte Carlo iteration.
PARTITIONS_TO_EXTRACT = ("test",)

BATCH_SIZE = 256
NUM_WORKERS = 2
PIN_MEMORY = True

# A profile can appear in test sets from multiple Monte Carlo repetitions.
# Keep all rows and preserve `random_split` as a metadata column.
# Do not combine raw latent axes across independently trained checkpoints
# into one UMAP without model-specific alignment or a shared final refit.

# Optional: save original gene-token tensors only for a small, specified audit set.
SAVE_GENE_TOKEN_TENSORS = False
MAX_GENE_TOKEN_PROFILES_PER_SPLIT = 5000
RANDOM_SEED = 66


# Full-tensor storage configuration.
# Default is held-out test profiles. Set ("train", "valid", "test") only after
# confirming available disk space; full tensors are large.
STORE_DTYPE_NAME = 'float16'
PROFILE_DTYPE_NAME = 'float32'
N_GENES = 978
LATENT_DIM = 32

# Approximate storage PER 100,000 profiles:
# one raw tensor = 100,000 x 978 x 32 x 2 bytes ≈ 5.83 GiB
# baseline + perturbed raw tensors ≈ 11.66 GiB
# baseline + perturbed 978-long pooled profiles ≈ 0.73 GiB (float32)


In [2]:
# -----------------------------------------------------------------------------
# 1. Imports and reproducibility.
# -----------------------------------------------------------------------------
import gc
import math
import random
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore", category=FutureWarning)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Output directory:", OUT_DIR)


Device: cuda
Output directory: /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/analysis/CellEmbeddings/gene_token_latents_random_split


In [3]:
# -----------------------------------------------------------------------------
# 2. Read data exactly as in training and apply the same sample-wise normalization.
# -----------------------------------------------------------------------------
for p in [DATA_PATH, LLM_PATH, MFP_PATH, TARGET_PATH, MODEL_DIR]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required path: {p}")

adata = sc.read(DATA_PATH)
gptEmbed_df = pd.read_csv(LLM_PATH, index_col=0)
MFP_df = pd.read_csv(MFP_PATH, index_col=0)
drug_targets = pd.read_csv(TARGET_PATH, index_col=0)

# Matches the original training script.
sc.pp.normalize_total(adata)

print(adata)
print("Number of genes:", adata.n_vars)
print("Available random split columns:", [c for c in RANDOM_SPLITS if c in adata.obs.columns])


AnnData object with n_obs × n_vars = 883077 × 978
    obs: 'cell_id', 'det_plate', 'det_well', 'lincs_phase', 'pert_dose', 'pert_dose_unit', 'pert_id', 'pert_iname', 'pert_mfc_id', 'pert_time', 'pert_time_unit', 'pert_type', 'rna_plate', 'rna_well', 'condition', 'cell_type', 'dose', 'cov_drug_dose_name', 'cov_drug_name', 'control', 'canonical_smiles', 'SMILES', 'paired_control_index', 'plate', 'random_split1', 'cell_split1', 'drug_split1', 'random_split2', 'cell_split2', 'drug_split2', 'random_split3', 'cell_split3', 'drug_split3', 'random_split4', 'cell_split4', 'drug_split4', 'random_split5', 'cell_split5', 'drug_split5'
Number of genes: 978
Available random split columns: ['random_split1', 'random_split2', 'random_split3', 'random_split4', 'random_split5']


In [4]:
# -----------------------------------------------------------------------------
# 3. Dataset. `return_metadata=True` retains the exact perturbation profile ID.
# -----------------------------------------------------------------------------
class GenePerturbationDataset(Dataset):
    def __init__(
        self,
        adata,
        gptEmbed_df,
        MFP_df,
        drug_targets,
        split_strategy,
        split_value,
        return_metadata=True,
    ):
        self.adata = adata
        self.gptEmbed_df = gptEmbed_df
        self.MFP_df = MFP_df
        self.drug_targets = drug_targets
        self.return_metadata = return_metadata
        self.indices = np.where(adata.obs[split_strategy].astype(str).values == str(split_value))[0]

        ctrl_mask = adata.obs["control"].values == 1
        X_ctrl = adata.X[ctrl_mask]
        X_ctrl = X_ctrl.toarray() if hasattr(X_ctrl, "toarray") else np.asarray(X_ctrl)
        cell_ids_ctrl = adata.obs.loc[ctrl_mask, "cell_id"].astype(str).values

        self.cell_stats = {}
        for cid in np.unique(cell_ids_ctrl):
            mat = X_ctrl[cell_ids_ctrl == cid]
            mu = mat.mean(axis=0, dtype=np.float32)
            var = mat.var(axis=0, dtype=np.float32)
            self.cell_stats[cid] = torch.from_numpy(np.concatenate([mu, var]).astype(np.float32))

    def __len__(self):
        return len(self.indices)

    @staticmethod
    def _dense_row(x):
        if hasattr(x, "toarray"):
            return np.asarray(x.toarray()).ravel()
        return np.asarray(x).ravel()

    def __getitem__(self, idx):
        real_idx = int(self.indices[idx])
        obs_row = self.adata.obs.iloc[real_idx]

        y_true = self._dense_row(self.adata.X[real_idx])

        paired_id = obs_row["paired_control_index"]
        paired_idx = int(self.adata.obs.index.get_loc(paired_id))
        x_baseline = self._dense_row(self.adata.X[paired_idx])

        cell_id = str(self.adata.obs.iloc[paired_idx]["cell_id"])
        cell_stats = self.cell_stats[cell_id]

        drug_name = obs_row["pert_iname"]
        x_drug_gpt = self.gptEmbed_df.loc[drug_name].values.astype(np.float32)
        x_drug_mfp = self.MFP_df.loc[drug_name].values.astype(np.float32)
        x_drug_targets = self.drug_targets.loc[drug_name].values.astype(np.float32)

        dose = float(obs_row["dose"])
        pert_time = float(obs_row["pert_time"])
        dose_feat = torch.tensor(math.log10(dose + 1.0), dtype=torch.float32)
        time_feat = torch.tensor(math.log10(pert_time + 1.0), dtype=torch.float32)

        tensors = (
            torch.tensor(x_baseline, dtype=torch.float32),
            cell_stats,
            torch.tensor(x_drug_gpt, dtype=torch.float32),
            torch.tensor(x_drug_mfp, dtype=torch.float32),
            torch.tensor(x_drug_targets, dtype=torch.float32),
            torch.tensor(y_true, dtype=torch.float32),
            dose_feat,
            time_feat,
        )

        if not self.return_metadata:
            return tensors

        metadata = {
            "obs_name": str(self.adata.obs_names[real_idx]),
            "adata_index": real_idx,
            "cell_id": str(obs_row["cell_id"]),
            "plate": str(obs_row["plate"]),
            "pert_iname": str(drug_name),
            "dose": dose,
            "pert_time": pert_time,
            "paired_control_index": str(paired_id),
            "paired_control_adata_index": paired_idx,
        }
        return tensors, metadata


In [5]:
# -----------------------------------------------------------------------------
# 4. DEPICT model copied from the training script, with optional latent returns.
#    Architecture/hyperparameters match the trained checkpoints exactly.
# -----------------------------------------------------------------------------
class Enc3Layer(nn.Module):
    def __init__(self, in_dim=512, latent_dim=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.GELU(), nn.BatchNorm1d(256),
            nn.Dropout(dropout),
            nn.Linear(256, 128), nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, latent_dim),
        )

    def forward(self, x):
        return self.net(x)


class GenePerturbationTransformer(nn.Module):
    def __init__(
        self,
        d_model=32,
        num_heads=8,
        num_encoder_layers=4,
        dropout=0.1,
        max_len=978,
        d_hidden=64,
        n_genes_target=1225,
        fp_bits=512,
        llm_dim=512,
        ae_latent=128,
    ):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.max_len = max_len

        d_mid = d_model // 2
        self.W1_gene = nn.Parameter(torch.empty(max_len, 3, d_mid))
        self.b1_gene = nn.Parameter(torch.empty(max_len, d_mid))
        self.W2_gene = nn.Parameter(torch.empty(max_len, d_mid, d_model))
        self.b2_gene = nn.Parameter(torch.empty(max_len, d_model))
        nn.init.xavier_uniform_(self.W1_gene)
        nn.init.zeros_(self.b1_gene)
        nn.init.xavier_uniform_(self.W2_gene)
        nn.init.zeros_(self.b2_gene)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dropout=dropout,
        )
        self.gene_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers,
        )

        self.ae_llm = Enc3Layer(llm_dim, latent_dim=ae_latent, dropout=dropout)
        self.ae_fp = Enc3Layer(fp_bits, latent_dim=ae_latent, dropout=dropout)

        def mlp_proj(in_dim, out_dim):
            return nn.Sequential(
                nn.Linear(in_dim, out_dim)
            )

        self.llm_proj_k = mlp_proj(ae_latent, num_heads * d_model)
        self.llm_proj_v = mlp_proj(ae_latent, num_heads * d_model)
        self.fp_proj_k  = mlp_proj(ae_latent, num_heads * d_model)
        self.fp_proj_v  = mlp_proj(ae_latent, num_heads * d_model)

        self.xattn1 = nn.MultiheadAttention(d_model, num_heads, dropout)
        self.xattn2 = nn.MultiheadAttention(d_model, num_heads, dropout)
        self.xattn3 = nn.MultiheadAttention(d_model, num_heads, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

        self.shared_ffn = nn.Sequential(
            nn.Linear(d_model, d_hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_hidden, d_model),
        )

        self.gamma_gene = nn.Parameter(torch.ones(max_len, d_model))
        self.beta_gene = nn.Parameter(torch.zeros(max_len, d_model))
        self.var_gamma = nn.Parameter(torch.zeros(max_len, d_model))
        self.var_beta = nn.Parameter(torch.zeros(max_len, d_model))

        self.W = nn.Parameter(torch.empty(max_len, d_model))
        self.b = nn.Parameter(torch.empty(max_len))
        self.Wmlp1 = nn.Parameter(torch.empty(max_len, d_model, d_mid))
        self.bmlp1 = nn.Parameter(torch.zeros(max_len, d_mid))
        self.Wmlp2 = nn.Parameter(torch.empty(max_len, d_mid))
        self.bmlp2 = nn.Parameter(torch.zeros(max_len))

        for p in (self.W, self.Wmlp1, self.Wmlp2):
            nn.init.xavier_uniform_(p)
        for p in (self.b, self.bmlp1, self.bmlp2):
            nn.init.zeros_(p)

        def _make_gate():
            gate = nn.Sequential(
                nn.Linear(2, 16), nn.GELU(),
                nn.Linear(16, 1), nn.Softplus(),
            )
            with torch.no_grad():
                gate[-2].weight.zero_()
                gate[-2].bias.fill_(-4.0)
            return gate

        self.gate_attn1 = _make_gate()
        self.gate_attn3 = _make_gate()

    def _kv(self, proj_k, proj_v, feat):
        batch_size = feat.size(0)
        k = proj_k(feat).view(batch_size, self.num_heads, self.d_model).permute(1, 0, 2)
        v = proj_v(feat).view(batch_size, self.num_heads, self.d_model).permute(1, 0, 2)
        return k, v

    def forward(
        self,
        x_base,
        cell_stats,
        x_llm,
        x_fp,
        x_tgt,
        dose_feat,
        time_feat,
        return_latents=False,
        return_gene_tokens=False,
    ):
        batch_size, n_genes = x_base.shape
        if n_genes != self.max_len:
            raise ValueError(f"Expected {self.max_len} genes, found {n_genes}.")

        mu = cell_stats[:, :n_genes]
        var = cell_stats[:, n_genes:]
        x_feat = torch.stack([x_base, mu, var], dim=-1)

        z_llm = self.ae_llm(x_llm)
        z_fp = self.ae_fp(x_fp)
        scalar_pair = torch.stack([dose_feat, time_feat], dim=-1)

        h1 = torch.einsum("bsi,sid->bsd", x_feat, self.W1_gene) + self.b1_gene
        h1 = torch.nn.functional.gelu(h1)
        x = torch.einsum("bsd,sdk->bsk", h1, self.W2_gene) + self.b2_gene
        x = x.transpose(0, 1)                       # (978, B, 32)
        x = self.gene_encoder(x)                     # baseline latent gene tokens
        baseline_gene_latent = x.transpose(0, 1)     # (B, 978, 32)

        k1, v1 = self._kv(self.llm_proj_k, self.llm_proj_v, z_llm)
        out, _ = self.xattn1(query=x, key=k1, value=v1)
        g1 = self.gate_attn1(scalar_pair).unsqueeze(0)
        x = self.norm1(x + self.dropout(g1 * out))

        k3, v3 = self._kv(self.fp_proj_k, self.fp_proj_v, z_fp)
        out, _ = self.xattn3(query=x, key=k3, value=v3)
        g3 = self.gate_attn3(scalar_pair).unsqueeze(0)
        x = self.norm3(x + self.dropout(g3 * out))
        perturbed_gene_latent = x.transpose(0, 1)    # (B, 978, 32)

        h = self.shared_ffn(perturbed_gene_latent)
        var_unsq = var.unsqueeze(-1)
        gamma_v = 1 + var_unsq * self.var_gamma.unsqueeze(0)
        beta_v = var_unsq * self.var_beta.unsqueeze(0)
        h = gamma_v * h + beta_v
        h = self.gamma_gene.unsqueeze(0) * h + self.beta_gene.unsqueeze(0)

        linear_out = torch.einsum("bsd,sd->bs", h, self.W) + self.b
        h_m = torch.einsum("bsd,sdm->bsm", h, self.Wmlp1) + self.bmlp1
        h_m = torch.nn.functional.gelu(h_m)
        mlp_out = torch.einsum("bsm,sm->bs", h_m, self.Wmlp2) + self.bmlp2
        prediction = linear_out + mlp_out

        if return_latents:
            output = {
                "prediction": prediction,
                "baseline_cell_latent": baseline_gene_latent.mean(dim=1),
                "perturbed_cell_latent": perturbed_gene_latent.mean(dim=1),
            }
            if return_gene_tokens:
                output["baseline_gene_latent"] = baseline_gene_latent
                output["perturbed_gene_latent"] = perturbed_gene_latent
            return output
        return prediction


In [6]:
# -----------------------------------------------------------------------------
# 5. Checkpoint utilities.
# -----------------------------------------------------------------------------
def checkpoint_path_for(split_name: str) -> Path:
    return MODEL_DIR / (
        f"transformer_d32h8l4_{split_name}_dp1_MSECor_lr1_CosSche_"
        "3XAttn_sepGene2EncPred_newAttn_FiLM_Enc3DimReducLa128_"
        "CellAware_frontEndMLPsimp_whole_DoseTimeAsScalarXattns_"
        "LLMMFP_first50epoch.pth"
    )

def load_model_for_split(split_name: str) -> tuple[nn.Module, Path]:
    ckpt_path = checkpoint_path_for(split_name)
    if not ckpt_path.exists():
        raise FileNotFoundError(
            f"Checkpoint missing for {split_name}:\n{ckpt_path}\n"
            "Update `checkpoint_path_for()` if the trained filename differs."
        )

    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    state_dict = checkpoint["model_state"] if isinstance(checkpoint, dict) and "model_state" in checkpoint else checkpoint

    model = GenePerturbationTransformer(
        d_model=32,
        num_heads=8,
        num_encoder_layers=4,
        dropout=0.1,
        max_len=978,
        d_hidden=64,
        n_genes_target=1225,
        fp_bits=512,
        llm_dim=512,
        ae_latent=128,
    ).to(DEVICE)

    incompatible = model.load_state_dict(state_dict, strict=True)
    if incompatible.missing_keys or incompatible.unexpected_keys:
        raise RuntimeError(
            f"Unexpected state-dict mismatch for {split_name}: {incompatible}"
        )
    model.eval()
    return model, ckpt_path

for split_name in RANDOM_SPLITS:
    print(split_name, "->", checkpoint_path_for(split_name).exists())


random_split1 -> True
random_split2 -> True
random_split3 -> True
random_split4 -> True
random_split5 -> True


In [7]:

# -----------------------------------------------------------------------------
# 6. Stream baseline, observed perturbed, and DEPICT-predicted perturbed
#    expression profiles to disk for every requested random split.
# -----------------------------------------------------------------------------
from numpy.lib.format import open_memmap

N_GENES = 978

def metadata_batch_to_frame(metadata: dict) -> pd.DataFrame:
    out = {}
    for key, values in metadata.items():
        if isinstance(values, torch.Tensor):
            out[key] = values.detach().cpu().numpy()
        else:
            out[key] = list(values)
    return pd.DataFrame(out)

def _paths(split_name: str, partition: str) -> dict:
    prefix = f"{split_name}_{partition}"
    return {
        "baseline": OUT_DIR / f"{prefix}_matched_baseline_expression_978_float32.npy",
        "observed": OUT_DIR / f"{prefix}_observed_perturbed_expression_978_float32.npy",
        "predicted": OUT_DIR / f"{prefix}_predicted_perturbed_expression_logits_978_float32.npy",
        "metadata": OUT_DIR / f"{prefix}_prediction_metadata.parquet",
    }

@torch.inference_mode()
def extract_predictions(model, split_name: str, partition: str):
    """
    Saves values in exact metadata row order:
      baseline[i, :]  : paired DMSO control expression used as x_base for perturbation i
      observed[i, :]  : observed drug-treated expression y_true
      predicted[i, :] : DEPICT output logits / predicted perturbed expression
    """
    dataset = GenePerturbationDataset(
        adata=adata,
        gptEmbed_df=gptEmbed_df,
        MFP_df=MFP_df,
        drug_targets=drug_targets,
        split_strategy=split_name,
        split_value=partition,
        return_metadata=True,
    )
    n_profiles = len(dataset)
    if n_profiles == 0:
        raise ValueError(f"No profiles found for {split_name} / {partition}.")

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY and DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
    )
    paths = _paths(split_name, partition)
    baseline_mm = open_memmap(paths["baseline"], mode="w+", dtype=np.float32, shape=(n_profiles, N_GENES))
    observed_mm = open_memmap(paths["observed"], mode="w+", dtype=np.float32, shape=(n_profiles, N_GENES))
    predicted_mm = open_memmap(paths["predicted"], mode="w+", dtype=np.float32, shape=(n_profiles, N_GENES))

    metadata_frames = []
    start = 0
    for tensors, metadata in loader:
        x_base, cell_stats, x_llm, x_mfp, x_targets, y_true, dose_feat, time_feat = tensors
        x_base = x_base.to(DEVICE, non_blocking=True)
        cell_stats = cell_stats.to(DEVICE, non_blocking=True)
        x_llm = x_llm.to(DEVICE, non_blocking=True)
        x_mfp = x_mfp.to(DEVICE, non_blocking=True)
        x_targets = x_targets.to(DEVICE, non_blocking=True)
        dose_feat = dose_feat.to(DEVICE, non_blocking=True)
        time_feat = time_feat.to(DEVICE, non_blocking=True)

        logits = model(
            x_base=x_base,
            cell_stats=cell_stats,
            x_llm=x_llm,
            x_fp=x_mfp,
            x_tgt=x_targets,
            dose_feat=dose_feat,
            time_feat=time_feat,
        )
        b = x_base.detach().cpu().numpy().astype(np.float32, copy=False)
        y = y_true.detach().cpu().numpy().astype(np.float32, copy=False)
        pred = logits.detach().cpu().numpy().astype(np.float32, copy=False)
        if b.shape != y.shape or y.shape != pred.shape or b.shape[1] != N_GENES:
            raise RuntimeError(f"Unexpected batch shapes baseline={b.shape}, observed={y.shape}, predicted={pred.shape}")

        stop = start + b.shape[0]
        baseline_mm[start:stop] = b
        observed_mm[start:stop] = y
        predicted_mm[start:stop] = pred

        batch_meta = metadata_batch_to_frame(metadata)
        batch_meta["random_split"] = split_name
        batch_meta["partition"] = partition
        batch_meta["row_in_prediction_file"] = np.arange(start, stop, dtype=np.int64)
        metadata_frames.append(batch_meta)
        start = stop

    if start != n_profiles:
        raise RuntimeError(f"Wrote {start:,} profiles; expected {n_profiles:,}.")
    baseline_mm.flush(); observed_mm.flush(); predicted_mm.flush()
    metadata_df = pd.concat(metadata_frames, ignore_index=True)
    metadata_df.to_parquet(paths["metadata"], index=False)
    del baseline_mm, observed_mm, predicted_mm

    return {
        "n_profiles": n_profiles,
        "n_cells": metadata_df["cell_id"].nunique(),
        "n_plates": metadata_df["plate"].nunique(),
        **{f"{k}_path": str(v) for k,v in paths.items()},
    }


In [8]:

# -----------------------------------------------------------------------------
# 7. Run prediction extraction for all requested random-split checkpoints.
# -----------------------------------------------------------------------------
manifest_rows = []
for split_name in RANDOM_SPLITS:
    model, ckpt_path = load_model_for_split(split_name)
    print(f"\nLoaded {split_name}: {ckpt_path.name}")
    for partition in PARTITIONS_TO_EXTRACT:
        summary = extract_predictions(model, split_name, partition)
        manifest_rows.append({
            "random_split": split_name,
            "partition": partition,
            "checkpoint_path": str(ckpt_path),
            **summary,
        })
        print(
            f"  {partition}: {summary['n_profiles']:,} profiles; "
            f"{summary['n_cells']} cells; {summary['n_plates']} plates"
        )
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

manifest = pd.DataFrame(manifest_rows)
manifest_path = OUT_DIR / "random_split_prediction_manifest.csv"
manifest.to_csv(manifest_path, index=False)
print("Saved manifest:", manifest_path)
display(manifest)


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(



Loaded random_split1: transformer_d32h8l4_random_split1_dp1_MSECor_lr1_CosSche_3XAttn_sepGene2EncPred_newAttn_FiLM_Enc3DimReducLa128_CellAware_frontEndMLPsimp_whole_DoseTimeAsScalarXattns_LLMMFP_first50epoch.pth
  test: 83,665 profiles; 82 cells; 2869 plates


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(



Loaded random_split2: transformer_d32h8l4_random_split2_dp1_MSECor_lr1_CosSche_3XAttn_sepGene2EncPred_newAttn_FiLM_Enc3DimReducLa128_CellAware_frontEndMLPsimp_whole_DoseTimeAsScalarXattns_LLMMFP_first50epoch.pth
  test: 83,665 profiles; 82 cells; 2868 plates


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


...

/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(



Loaded random_split4: transformer_d32h8l4_random_split4_dp1_MSECor_lr1_CosSche_3XAttn_sepGene2EncPred_newAttn_FiLM_Enc3DimReducLa128_CellAware_frontEndMLPsimp_whole_DoseTimeAsScalarXattns_LLMMFP_first50epoch.pth
  test: 83,665 profiles; 82 cells; 2867 plates


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(



Loaded random_split5: transformer_d32h8l4_random_split5_dp1_MSECor_lr1_CosSche_3XAttn_sepGene2EncPred_newAttn_FiLM_Enc3DimReducLa128_CellAware_frontEndMLPsimp_whole_DoseTimeAsScalarXattns_LLMMFP_first50epoch.pth
  test: 83,665 profiles; 82 cells; 2873 plates
Saved manifest: /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/analysis/CellEmbeddings/gene_token_latents_random_split/random_split_prediction_manifest.csv


,random_split,partition,checkpoint_path,n_profiles,n_cells,n_plates,baseline_path,observed_path,predicted_path,metadata_path
0,random_split1,test,/work/users/m/e/meisheng/Dissertation/Experime...,83665,82,2869,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...
1,random_split2,test,/work/users/m/e/meisheng/Dissertation/Experime...,83665,82,2868,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...
2,random_split3,test,/work/users/m/e/meisheng/Dissertation/Experime...,83665,82,2871,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...
3,random_split4,test,/work/users/m/e/meisheng/Dissertation/Experime...,83665,82,2867,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...
4,random_split5,test,/work/users/m/e/meisheng/Dissertation/Experime...,83665,82,2873,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...,/work/users/m/e/meisheng/Dissertation/Experime...


In [9]:

# -----------------------------------------------------------------------------
# 8. Verify stored arrays and matching metadata for one split.
# -----------------------------------------------------------------------------
CHECK_SPLIT = RANDOM_SPLITS[0]
CHECK_PARTITION = PARTITIONS_TO_EXTRACT[0]
p = _paths(CHECK_SPLIT, CHECK_PARTITION)
meta = pd.read_parquet(p["metadata"])
x_base = np.load(p["baseline"], mmap_mode="r")
y_obs = np.load(p["observed"], mmap_mode="r")
y_pred = np.load(p["predicted"], mmap_mode="r")

print("Metadata:", meta.shape)
print("Matched baseline:", x_base.shape, x_base.dtype)
print("Observed perturbed:", y_obs.shape, y_obs.dtype)
print("Predicted perturbed logits:", y_pred.shape, y_pred.dtype)
assert len(meta) == x_base.shape[0] == y_obs.shape[0] == y_pred.shape[0]
assert x_base.shape[1] == y_obs.shape[1] == y_pred.shape[1] == N_GENES
print(meta[["obs_name", "paired_control_index", "paired_control_adata_index", "cell_id", "plate"]].head())


Metadata: (83665, 12)
Matched baseline: (83665, 978) float32
Observed perturbed: (83665, 978) float32
Predicted perturbed logits: (83665, 978) float32
                         obs_name            paired_control_index  \
0  REP.A001_A375_24H_X1_B22:C13-2  REP.A001_A375_24H_X1_B22:J16-2   
1  REP.A001_A375_24H_X1_B22:C15-2  REP.A001_A375_24H_X1_B22:F09-2   
2  REP.A001_A375_24H_X1_B22:C19-2  REP.A001_A375_24H_X1_B22:J14-2   
3  REP.A001_A375_24H_X1_B22:C24-2  REP.A001_A375_24H_X1_B22:F11-2   
4  REP.A001_A375_24H_X1_B22:D17-2  REP.A001_A375_24H_X1_B22:B03-2   

   paired_control_adata_index cell_id                     plate  
0                      836666    A375  REP.A001_A375_24H_X1_B22  
1                      836659    A375  REP.A001_A375_24H_X1_B22  
2                      836664    A375  REP.A001_A375_24H_X1_B22  
3                      836661    A375  REP.A001_A375_24H_X1_B22  
4                      836653    A375  REP.A001_A375_24H_X1_B22  
